<a href="https://colab.research.google.com/github/donoftime2018/Mental-Health-Chatbot/blob/LLama-3.2/Llama_3_2ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import userdata
userdata.get('HF_TOKEN')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -U sentence-transformers
!pip install -q bitsandbytes>=0.46.1
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer, BitsAndBytesConfig
import pandas as pd
import torch
import re
from sklearn.metrics.pairwise import cosine_similarity
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sentence_transformers import SentenceTransformer
from peft import LoraConfig, TaskType

In [ ]:
device = torch.device("cuda")
device

In [ ]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
)

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)


In [ ]:
ogModel = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-3.2-3B",
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True
)

ogModel.to(device)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-3B")
tokenizer.pad_token = tokenizer.eos_token
tokenizer.chat_template = "{% for message in messages %}\n{% if message['role'] == 'user' %}\n{{ '<|user|>\n' + message['content'] + eos_token }}\n{% elif message['role'] == 'system' %}\n{{ '<|system|>\n' + message['content'] + eos_token }}\n{% elif message['role'] == 'assistant' %}\n{{ '<|assistant|>\n'  + message['content'] + eos_token }}\n{% endif %}\n{% if loop.last and add_generation_prompt %}\n{{ '<|assistant|>' }}\n{% endif %}\n{% endfor %}"
tokenizer

In [ ]:
dataset = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/normalized_context_and_response.csv")

In [ ]:
def remove_special_tokens(text):
    text = re.sub(r'<\|.*?\|>', '', text)
    return text.strip()

In [ ]:
def combineText(example):
  messages = [
      {"role": "user", "content": example['contexts']},
      {"role": "assistant", "content": example['responses']}
  ]
  formatted_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
  return {"text": formatted_text}

In [ ]:
def encode(example):
  return tokenizer(example['text'], truncation=True, padding=True, max_length=128)

In [ ]:
def add_labels(example):
    example['labels']=example['input_ids']
    return example

In [ ]:
contexts = dataset['context'].apply(remove_special_tokens).astype("str").values
contexts[:1]

In [ ]:
responses = dataset['response'].astype("str").apply(remove_special_tokens).values
responses[:1]

In [ ]:
datasets = Dataset.from_dict({
    "contexts": contexts,
    "responses": responses
})
datasets

In [ ]:
datasets_split = datasets.train_test_split( test_size=0.1)
datasets_split

In [ ]:
trainSet = datasets_split['train']
trainSet

In [ ]:
testSet = datasets_split['test']
testSet

In [ ]:
trainSet = trainSet.map(combineText)

In [ ]:
testSet = testSet.map(combineText)

In [ ]:
trainSet = trainSet.map(encode, batched=True)

In [ ]:
testSet = testSet.map(encode, batched=True)

In [ ]:
trainSet = trainSet.map(add_labels)
trainSet

In [ ]:
testSet = testSet.map(add_labels)
testSet

In [ ]:
trainingArgs = TrainingArguments(
    max_steps=60,
    num_train_epochs=2,
    learning_rate=2e-4,
    per_device_train_batch_size=2, # Further reduced batch size to prevent OOM
    per_device_eval_batch_size=2, # Reduced eval batch size for consistency
    gradient_accumulation_steps=1, # Use gradient accumulation to achieve an effective batch size of 1 * 8 = 8
    eval_strategy='steps',
    weight_decay=0.01,
    fp16=False,
    bf16=True, # Use bfloat16 for better memory stability and efficiency on T4
    gradient_checkpointing=True, # Enable gradient checkpointing to save memory
    max_grad_norm = 1.0
)

In [ ]:
ogModel.add_adapter(lora_config, adapter_name="my_adapter")

In [ ]:
trainer = Trainer(
    model=ogModel,
    args=trainingArgs,
    train_dataset=trainSet,#.select(range(100)),  # Using a small subset for faster training
    eval_dataset=testSet#.select(range(80))    # Using a small subset for faster evaluation
)

In [ ]:
trainer.evaluate(testSet)

In [ ]:
# trainer.predict(testSet)

In [ ]:
trainer.train()

In [ ]:
trainer.save_model("/content/drive/MyDrive/Colab Notebooks/Llama 3.2 Fine Tuned (Full train+test sets)")

In [ ]:
trainer.evaluate(testSet)#.select(range(80)))

In [ ]:
# trainer.predict(testSet)#.select(range(80)))